# NOURA EL KHOLTI

## L'algorithme DBSCAN

DBSCAN est un algorithme de clustering basé sur la densité. Contrairement à d'autres méthodes, il regroupe les points qui sont proches les uns des autres dans des zones denses et identifie automatiquement les points isolés comme étant du bruit.

### 1. Initialisation des données

In [ ]:
import numpy as np
import pandas as pd

# Création du jeu de données
donnees = {
    'Points': ['P1', 'P2', 'P3', 'P4', 'P5'],
    'X': [5, 5.5, 12, 12.5, 25],
    'Y': [5, 6  , 12, 11  , 40]
}

df = pd.DataFrame(donnees)
points_array = df[['X', 'Y']].values

print("Jeu de données initialisé :")
print(df)

```
Jeu de données initialisé :
  Points     X   Y
0     P1   5.0   5
1     P2   5.5   6
2     P3  12.0  12
3     P4  12.5  11
4     P5  25.0  40
```

### 2. Fonctions de Distance et de Voisinage

Le fonctionnement de DBSCAN repose sur la distance euclidienne pour identifier les points proches selon un rayon donné eps (ε).

In [ ]:
def distance_euclidienne(p1, p2):
    """Calcule la distance entre deux points dans un espace 2D."""
    return np.sqrt(np.sum((p1 - p2) ** 2))

def trouver_voisins(donnees, index_point, eps):
    """Retourne les indices de tous les points situés à une distance inférieure à 'eps'."""
    voisins = []
    for i in range(len(donnees)):
        if distance_euclidienne(donnees[index_point], donnees[i]) <= eps:
            voisins.append(i)
    return voisins

### 3. L'Algorithme DBSCAN

Cette fonction parcourt les données, identifie les "points cœurs" (core points) et étend les clusters. Les points qui ne répondent pas aux critères de densité sont marqués comme du bruit.

In [ ]:
def dbscan_sur_mesure(donnees, eps, min_pts):
    """
    Implémentation de l'algorithme DBSCAN.
    Retourne une liste de labels (0: non visité, -1: bruit, 1+: ID du cluster).
    """
    n_points = len(donnees)
    labels = [0] * n_points
    cluster_id = 0
    
    for i in range(n_points):
        # Si le point a déjà été traité, on passe au suivant
        if labels[i] != 0:
            continue
            
        # Recherche des voisins
        voisins = trouver_voisins(donnees, i, eps)
        
        # Vérification de la densité (Bruit ou Point Cœur)
        if len(voisins) < min_pts:
            labels[i] = -1
        else:
            # C'est un point cœur ! On commence un nouveau cluster
            cluster_id += 1
            labels[i] = cluster_id
            
            # Expansion du cluster à partir des voisins
            file_recherche = voisins.copy()
            if i in file_recherche:
                file_recherche.remove(i)
                
            idx = 0
            while idx < len(file_recherche):
                voisin_idx = file_recherche[idx]
                
                # Si le point était marqué comme bruit, il devient un point de bordure
                if labels[voisin_idx] == -1:
                    labels[voisin_idx] = cluster_id
                
                # Si le point n'a jamais été visité
                elif labels[voisin_idx] == 0:
                    labels[voisin_idx] = cluster_id
                    
                    # Si ce voisin est aussi un point cœur, on ajoute ses voisins à la file
                    nouveaux_voisins = trouver_voisins(donnees, voisin_idx, eps)
                    if len(nouveaux_voisins) >= min_pts:
                        for n in nouveaux_voisins:
                            if n not in file_recherche:
                                file_recherche.append(n)
                idx += 1
                
    return labels

### 4. Exécution et Résultats

Nous appliquons ici les paramètres eps = 1.9 et MinPts = 4.

In [ ]:
# Paramètres de l'algorithme
EPSILON = 1.9
MIN_SAMPLES = 4

# Lancement de DBSCAN
resultats = dbscan_sur_mesure(points_array, EPSILON, MIN_SAMPLES)

# Mise en forme pour l'affichage
labels_finaux = [f"Cluster {r}" if r > 0 else "Bruit (Noise)" for r in resultats]
df['Resultat_Clustering'] = labels_finaux

print("\n--- Résultats du Clustering ---")
print(df)

# Résumé statistique
nb_clusters = len(set(resultats)) - (1 if -1 in resultats else 0)
nb_bruit = list(resultats).count(-1)

print(f"\nNombre de clusters trouvés : {nb_clusters}")
print(f"Nombre de points considérés comme du bruit : {nb_bruit}")

```
--- Résultats du Clustering ---
  Points     X   Y Resultat_Clustering
0     P1   5.0   5       Bruit (Noise)
1     P2   5.5   6       Bruit (Noise)
2     P3  12.0  12       Bruit (Noise)
3     P4  12.5  11       Bruit (Noise)
4     P5  25.0  40       Bruit (Noise)

Nombre de clusters trouvés : 0
Nombre de points considérés comme du bruit : 5
```